# Day 2 — EDA for Music: Catalog Coherence & Release Cadence

**Course:** INFO 7390  
**Goal:** Detect fake/ghost artists on Spotify using two complementary EDA signals.

## What we're doing today
1. **Ingest** — Populate Neo4j with seed artist catalogs
2. **Signal 1: Catalog Coherence** — Do all tracks sound statistically identical? (PCA on audio features)
3. **Signal 2: Release Cadence** — Were tracks released in suspicious bursts? (Isolation Forest on gaps)
4. **Comparison** — Ghost vs Organic artists side-by-side
5. **Visualization** — Plots for report

## Why this works
Ghost artist factories generate tracks with an AI model locked to fixed parameters → every track has near-identical danceability, energy, valence, acousticness → **very low catalog variance**.

They also upload 50-100 tracks at once to maximize streaming revenue before detection → **burst cadence**.

Real artists experiment across their catalog and release incrementally.

In [ ]:
# Setup
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.api.spotify_client import SpotifyClient
from src.graph.neo4j_client import Neo4jClient
from src.utils.ground_truth import get_all_seeds, get_ghost_seeds, get_organic_seeds
from src.signals import signal1_catalog_coherence as s1
from src.signals import signal2_release_cadence as s2

# Clients
spotify = SpotifyClient()
neo4j = Neo4jClient()

# Verify connections
assert spotify.test_connection(), "Spotify connection failed"
assert neo4j.test_connection(), "Neo4j connection failed"
print("Connections OK")

## 0. Ingest — Populate Neo4j with seed catalogs

Run once (idempotent — safe to re-run, uses MERGE). Takes ~5 minutes for 5 organic artists.

In [ ]:
from src.ingest.seed_ingest import run as ingest_run

# Ingest only organic seeds first (they have confirmed IDs)
# Ghost seeds will be resolved by search during ingest
stats = ingest_run(dry_run=False)
print(stats)

In [ ]:
# Verify Neo4j node counts after ingest
counts = neo4j.count_nodes()
for label, count in counts.items():
    print(f"  {label:<20} {count:>6}")

## 1. Load Seed Catalog CSV

In [ ]:
catalog = pd.read_csv("../data/processed/seed_catalog.csv")
print(f"Total tracks: {len(catalog)}")
print(f"Artists: {catalog['artist_name'].nunique()}")
print(f"Ghost tracks: {catalog[catalog.is_ghost == True].shape[0]}")
print(f"Organic tracks: {catalog[catalog.is_ghost == False].shape[0]}")
catalog.groupby(["artist_name", "is_ghost"]).size().reset_index(name="track_count")

## 2. Signal 1 — Catalog Coherence (Exercise 1)

**Hypothesis:** Ghost artist catalogs have near-zero variance in 4D audio feature space.

**Method:**
1. Join track IDs with Kaggle audio features dataset
2. Compute covariance matrix eigenvalues
3. Run PCA — ghost artists should have PC1 ≈ 100% explained variance

In [ ]:
# Score all seeds for catalog coherence
all_seeds = get_all_seeds()
# Only score seeds that have a spotify_id
valid_seeds = [s for s in all_seeds if s.get("spotify_id")]

df_s1 = s1.score_seed_set(spotify, valid_seeds)
print(s1.summary_table(df_s1))

In [ ]:
# Visualization — Overall Variance by Artist
fig, ax = plt.subplots(figsize=(12, 5))

colors = ["#e74c3c" if row.is_ghost_label else "#2ecc71" for _, row in df_s1.iterrows()]
bars = ax.barh(df_s1["artist_name"], df_s1["overall_variance"], color=colors)

ax.axvline(x=s1.HIGH_SUSPICION_THRESHOLD, color="red", linestyle="--", alpha=0.7, label="HIGH threshold")
ax.axvline(x=s1.MEDIUM_SUSPICION_THRESHOLD, color="orange", linestyle="--", alpha=0.7, label="MEDIUM threshold")

ghost_patch = mpatches.Patch(color='#e74c3c', label='Ghost artist')
organic_patch = mpatches.Patch(color='#2ecc71', label='Organic artist')
ax.legend(handles=[ghost_patch, organic_patch, 
                   plt.Line2D([0],[0], color='red', linestyle='--', label='HIGH threshold'),
                   plt.Line2D([0],[0], color='orange', linestyle='--', label='MEDIUM threshold')])

ax.set_xlabel("Overall Variance (4D audio feature space)")
ax.set_title("Signal 1: Catalog Coherence — Ghost vs Organic Artists\n"
             "Lower variance = more suspicious (every track sounds the same)")
plt.tight_layout()
plt.savefig("../paper/signal1_catalog_coherence.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# PCA explained variance — ghost artists should cluster near PC1=100%
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=False)
axes = axes.flatten()

for i, (_, row) in enumerate(df_s1.iterrows()):
    ax = axes[i]
    explained = row["pca_explained_variance"]
    if explained:
        color = "#e74c3c" if row.is_ghost_label else "#2ecc71"
        ax.bar(range(1, len(explained) + 1), explained, color=color, alpha=0.8)
        ax.set_ylim(0, 1)
        ax.set_title(f"{row.artist_name}\n({'ghost' if row.is_ghost_label else 'organic'})",
                     fontsize=9)
        ax.set_xlabel("PC", fontsize=8)
        ax.set_ylabel("Explained var", fontsize=8)
    else:
        ax.text(0.5, 0.5, "No Kaggle data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{row.artist_name}\n(no data)", fontsize=9)

plt.suptitle("PCA Explained Variance by Artist\nGhost artists: PC1 dominates (low diversity)", y=1.02)
plt.tight_layout()
plt.savefig("../paper/signal1_pca.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Per-feature variance heatmap
feat_data = []
for _, row in df_s1.iterrows():
    vf = row["variance_per_feature"]
    if vf:
        feat_data.append({"artist": row["artist_name"], "label": "ghost" if row.is_ghost_label else "organic", **vf})

if feat_data:
    feat_df = pd.DataFrame(feat_data).set_index("artist")
    numeric_cols = [c for c in feat_df.columns if c != "label"]
    feat_df_num = feat_df[numeric_cols].astype(float)

    plt.figure(figsize=(10, 6))
    sns.heatmap(feat_df_num, annot=True, fmt=".3f", cmap="YlOrRd_r",
                linewidths=0.5, cbar_kws={"label": "Variance (lower = more suspicious)"})
    plt.title("Per-Feature Variance Heatmap\n(dark = low variance = ghost-like)")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig("../paper/signal1_feature_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

## 3. Signal 2 — Release Cadence (Isolation Forest)

**Hypothesis:** Ghost artists release in coordinated bursts — many tracks on the same day, then silence.

**Method:**
1. Get all album release dates per artist
2. Compute inter-release gaps (days)
3. Run Isolation Forest on gap sequence
4. Compute burst ratio (max fraction in any 7-day window)

In [ ]:
# Score all seeds for release cadence
df_s2 = s2.score_seed_set(spotify, valid_seeds)
display_cols = ["artist_name", "is_ghost_label", "total_releases", "max_single_day_releases",
                "burst_ratio", "single_day_fraction", "suspicion_level", "suspicion_score"]
print(df_s2[[c for c in display_cols if c in df_s2.columns]].to_string(index=False))

In [ ]:
# Visualization — Burst Ratio vs Catalog Variance (combined scatter)
# Merge both signals
merged = df_s1[["artist_id", "artist_name", "overall_variance", "is_ghost_label"]].merge(
    df_s2[["artist_id", "burst_ratio", "suspicion_score"]].rename(columns={"suspicion_score": "cadence_score"}),
    on="artist_id", how="inner"
)

fig = px.scatter(
    merged,
    x="overall_variance",
    y="burst_ratio",
    color="is_ghost_label",
    text="artist_name",
    color_discrete_map={True: "#e74c3c", False: "#2ecc71"},
    labels={
        "overall_variance": "Catalog Variance (Signal 1) — lower = more uniform",
        "burst_ratio": "Burst Ratio (Signal 2) — higher = more suspicious",
        "is_ghost_label": "Ghost Artist"
    },
    title="Signal 1 vs Signal 2: Ghost Artists Should Appear Bottom-Right",
    size_max=15,
)
fig.update_traces(textposition="top center")
fig.add_hline(y=0.6, line_dash="dash", line_color="orange", annotation_text="HIGH burst")
fig.add_vline(x=0.02, line_dash="dash", line_color="red", annotation_text="HIGH suspicion")
fig.show()
fig.write_html("../paper/signal1_vs_signal2.html")

In [ ]:
# Release timeline visualization per artist
from collections import Counter
import plotly.graph_objects as go

fig = go.Figure()

# Show top 5 artists by track count (mix of ghost + organic)
top5 = df_s2.nlargest(5, "total_releases")["artist_id"].tolist()

for _, row in df_s2[df_s2["artist_id"].isin(top5)].iterrows():
    date_counts = row.get("date_counts", {})
    if not date_counts:
        continue
    dates = sorted(date_counts.keys())
    counts = [date_counts[d] for d in dates]
    color = "#e74c3c" if row.is_ghost_label else "#2ecc71"
    fig.add_trace(go.Bar(
        x=dates, y=counts,
        name=f"{row.artist_name} ({'ghost' if row.is_ghost_label else 'organic'})",
        marker_color=color,
        opacity=0.7
    ))

fig.update_layout(
    title="Release Timeline — Releases per Day per Artist<br>"
          "<sup>Ghost artists: tall narrow spikes. Organic: gradual spread.</sup>",
    xaxis_title="Date",
    yaxis_title="Releases on that date",
    barmode="overlay",
    height=500,
)
fig.show()
fig.write_html("../paper/signal2_release_timeline.html")

In [ ]:
# Gap distribution — Isolation Forest anomaly visualization
fig, axes = plt.subplots(1, min(4, len(df_s2)), figsize=(16, 4))
if len(df_s2) == 1:
    axes = [axes]

for i, (_, row) in enumerate(df_s2.head(4).iterrows()):
    ax = axes[i]
    dates = row.get("release_dates", [])
    if len(dates) < 2:
        ax.text(0.5, 0.5, "<3 releases", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(row.artist_name)
        continue

    from datetime import datetime
    import numpy as np
    ordinals = sorted(datetime.fromisoformat(d).toordinal() for d in dates)
    gaps = np.diff(ordinals)

    color = "#e74c3c" if row.is_ghost_label else "#2ecc71"
    ax.hist(gaps, bins=20, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(x=np.median(gaps), color="navy", linestyle="--", label=f"median={np.median(gaps):.0f}d")
    ax.set_title(f"{row.artist_name}\n({'ghost' if row.is_ghost_label else 'organic'})", fontsize=9)
    ax.set_xlabel("Days between releases")
    ax.set_ylabel("Count")
    ax.legend(fontsize=7)

plt.suptitle("Inter-Release Gap Distribution\nGhost artists: all gaps near zero (burst release)")
plt.tight_layout()
plt.savefig("../paper/signal2_gap_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Combined Suspicion Score — Ghost vs Organic

In [ ]:
# Combined score: average of Signal 1 + Signal 2 suspicion
combined = df_s1[["artist_id", "artist_name", "is_ghost_label", "genre",
                   "overall_variance", "suspicion_score"]].rename(
    columns={"suspicion_score": "s1_score"}
).merge(
    df_s2[["artist_id", "suspicion_score", "burst_ratio"]].rename(
        columns={"suspicion_score": "s2_score"}
    ),
    on="artist_id",
    how="inner"
)
combined["combined_score"] = (combined["s1_score"] + combined["s2_score"]) / 2
combined["verdict"] = combined["combined_score"].apply(
    lambda x: "HIGH" if x >= 0.6 else ("MEDIUM" if x >= 0.35 else "LOW")
)
combined = combined.sort_values("combined_score", ascending=False)

# Display
display_cols = ["artist_name", "is_ghost_label", "s1_score", "s2_score", "combined_score", "verdict"]
print(combined[display_cols].to_string(index=False))

In [ ]:
# Final bar chart: combined suspicion score
fig, ax = plt.subplots(figsize=(12, 5))

colors = ["#e74c3c" if row.is_ghost_label else "#2ecc71" for _, row in combined.iterrows()]
bars = ax.barh(combined["artist_name"], combined["combined_score"], color=colors, alpha=0.85)

ax.axvline(x=0.6, color="red", linestyle="--", alpha=0.8, label="HIGH suspicion threshold")
ax.axvline(x=0.35, color="orange", linestyle="--", alpha=0.8, label="MEDIUM threshold")

for bar, (_, row) in zip(bars, combined.iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{row.combined_score:.2f} [{row.verdict}]",
            va="center", fontsize=9)

ghost_patch = mpatches.Patch(color='#e74c3c', label='Ghost artist')
organic_patch = mpatches.Patch(color='#2ecc71', label='Organic artist')
ax.legend(handles=[
    ghost_patch, organic_patch,
    plt.Line2D([0],[0], color='red', linestyle='--', label='HIGH threshold'),
    plt.Line2D([0],[0], color='orange', linestyle='--', label='MEDIUM threshold')
])
ax.set_xlim(0, 1.2)
ax.set_xlabel("Combined Suspicion Score (0–1)")
ax.set_title("Day 2 Results: Combined Suspicion Score (Signal 1 + Signal 2)\n"
             "Ghost artists expected to score > 0.6")
plt.tight_layout()
plt.savefig("../paper/day2_combined_score.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPlots saved to ../paper/")

## 5. Neo4j Graph State After Ingest

In [ ]:
# Node counts
counts = neo4j.count_nodes()
print("Neo4j graph state:")
for label, count in counts.items():
    print(f"  {label:<25} {count:>6}")

In [ ]:
# ISRC clusters — production companies with multiple artists
clusters = neo4j.get_isrc_clusters()
if clusters:
    print("Production company clusters (multiple artists sharing ISRC prefix):")
    for c in clusters:
        print(f"  {c['isrc_prefix']} ({c['company_name']}): {c['artist_count']} artists")
        print(f"    Artists: {', '.join(c['artists'])}")
else:
    print("No multi-artist ISRC clusters found yet (Day 3 will expand the search)")

## Summary

| Signal | What it measures | Ghost signature |
|--------|-----------------|------------------|
| **Signal 1: Catalog Coherence** | Variance of audio features (danceability, energy, valence, acousticness) | Overall variance < 0.02, PC1 ≈ 100% |
| **Signal 2: Release Cadence** | Burst ratio, inter-release gaps | burst_ratio > 0.6, max_single_day > 10 |

**Day 3 plan:** Exercise 3 — ISRC join to expose hidden production company ownership networks